# Phase 1 — Colab runtime check

This notebook verifies the repository commit, locked Python dependencies, NVIDIA runtime, PyTorch CUDA execution, writable project directories, and the shared environment report. It does **not** download datasets, load models, or train anything.

Before running, select **Runtime → Change runtime type → GPU**. The repository must be public, or Git authentication must be configured through a secure Colab mechanism. Never paste a token into this notebook.

## 1. Pin the repository revision

Replace `REPLACE_WITH_TESTED_COMMIT_SHA` with the full SHA produced by `git rev-parse HEAD` after the runtime-check implementation is committed and tested locally.

In [ ]:
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/pranavshrivastava1104/deep_learning_project.git"
GIT_SHA = "59010abe4d244d5f167e52d2509582303fc90fda"
PROJECT_DIRECTORY = Path("/content/deep_learning_project")

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f"This project requires Python 3.11, but Colab is running {sys.version.split()[0]}. "
        "Stop here and reconcile the compatibility policy before installing packages."
    )
if GIT_SHA == "REPLACE_WITH_TESTED_COMMIT_SHA" or len(GIT_SHA) < 7:
    raise ValueError(
        "Set GIT_SHA to the reviewed commit before running this notebook. "
        "Use `git rev-parse HEAD` in the tested local repository."
    )

print(f"Repository: {REPOSITORY_URL}")
print(f"Requested revision: {GIT_SHA}")
print(f"Checkout directory: {PROJECT_DIRECTORY}")

Repository: https://github.com/pranavshrivastava1104/deep_learning_project.git
Requested revision: c22d9bfae9620067925b3c5a8c81f074b86a0877
Checkout directory: /content/deep_learning_project


## 2. Clone and verify the exact Git commit

In [5]:
import os
import subprocess
import sys


def run(command: list[str], *, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    print("+", " ".join(command))
    return subprocess.run(command, cwd=cwd, check=True, text=True)


if not PROJECT_DIRECTORY.exists():
    run(["git", "clone", REPOSITORY_URL, str(PROJECT_DIRECTORY)])
elif not (PROJECT_DIRECTORY / ".git").is_dir():
    raise RuntimeError(f"{PROJECT_DIRECTORY} exists but is not a Git repository")
else:
    run(["git", "fetch", "--all", "--tags"], cwd=PROJECT_DIRECTORY)

run(["git", "checkout", "--detach", GIT_SHA], cwd=PROJECT_DIRECTORY)
EXPECTED_GIT_SHA = subprocess.check_output(
    ["git", "rev-parse", f"{GIT_SHA}^{{commit}}"], cwd=PROJECT_DIRECTORY, text=True
).strip()
ACTUAL_GIT_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_DIRECTORY, text=True
).strip()
if ACTUAL_GIT_SHA != EXPECTED_GIT_SHA:
    raise RuntimeError(
        f"Git verification failed: expected {EXPECTED_GIT_SHA}, got {ACTUAL_GIT_SHA}"
    )

os.chdir(PROJECT_DIRECTORY)
sys.path.insert(0, str(PROJECT_DIRECTORY))
print(f"✅ Git SHA matched: {ACTUAL_GIT_SHA}")

+ git fetch --all --tags
+ git checkout --detach c22d9bfae9620067925b3c5a8c81f074b86a0877
✅ Git SHA matched: c22d9bfae9620067925b3c5a8c81f074b86a0877


## 3. Install the explicit locked Colab environment

The hash-checked requirements export controls third-party packages. The editable project install uses `--no-deps`, preventing a second independent dependency resolution.

In [6]:
import importlib

run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--require-hashes",
        "-r",
        "requirements/colab.txt",
    ],
    cwd=PROJECT_DIRECTORY,
)
run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "."],
    cwd=PROJECT_DIRECTORY,
)
required_modules = ("torch", "torchvision", "onnx", "onnxruntime", "open_clip")
for module_name in required_modules:
    importlib.import_module(module_name)
    print(f"✅ Imported {module_name}")
print("✅ Locked requirements and project package installed")

+ /usr/bin/python3 -m pip install --require-hashes -r requirements/colab.txt
+ /usr/bin/python3 -m pip install --no-deps -e .
✅ Imported torch
✅ Imported torchvision
✅ Imported onnx
✅ Imported onnxruntime
✅ Imported open_clip
✅ Locked requirements and project package installed


## 4. Verify NVIDIA and PyTorch CUDA execution

Both checks matter: `nvidia-smi` proves that the runtime exposes an NVIDIA device, while the tensor operation proves that the installed PyTorch build can actually execute on it.

In [7]:
import shutil

import torch

if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "nvidia-smi is unavailable. Select Runtime > Change runtime type > GPU, "
        "then restart and rerun the notebook."
    )
run(["nvidia-smi"])

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch cannot use CUDA. Select Runtime > Change runtime type > GPU, "
        "then restart and rerun the notebook."
    )

GPU_NAME = torch.cuda.get_device_name(0)
tensor = torch.tensor([1.0, 2.0], device="cuda")
result = tensor * 2
torch.cuda.synchronize()
if result.cpu().tolist() != [2.0, 4.0]:
    raise RuntimeError(f"Unexpected CUDA tensor result: {result}")

print(f"✅ CUDA available through PyTorch {torch.__version__}")
print(f"✅ GPU operation successful on {GPU_NAME}: {result}")

+ nvidia-smi
✅ CUDA available through PyTorch 2.13.0+cu130
✅ GPU operation successful on Tesla T4: tensor([2., 4.], device='cuda:0')


## 5. Generate the shared environment report

In [8]:
REPORT_PATH = PROJECT_DIRECTORY / "artifacts" / "environment-colab.json"
report_environment = os.environ.copy()
report_environment.update(
    {
        "APP_ENV": "colab",
        "GIT_SHA": ACTUAL_GIT_SHA,
        "LOG_FORMAT": "json",
        "SERVICE_NAME": "environment-report",
    }
)
command = [
    sys.executable,
    "-m",
    "pipelines.environment_report",
    "--environment",
    "colab",
    "--require-cuda",
    "--repository-root",
    str(PROJECT_DIRECTORY),
    "--output",
    str(REPORT_PATH),
]
print("+", " ".join(command))
subprocess.run(
    command, cwd=PROJECT_DIRECTORY, env=report_environment, check=True, text=True
)
print(f"✅ Environment report created: {REPORT_PATH}")

+ /usr/bin/python3 -m pipelines.environment_report --environment colab --require-cuda --repository-root /content/deep_learning_project --output /content/deep_learning_project/artifacts/environment-colab.json


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pipelines.environment_report', '--environment', 'colab', '--require-cuda', '--repository-root', '/content/deep_learning_project', '--output', '/content/deep_learning_project/artifacts/environment-colab.json']' returned non-zero exit status 2.

## 6. Verify and display the result

ONNX Runtime providers are recorded but `CUDAExecutionProvider` is not required in Phase 1 because the current lock deliberately uses the portable `onnxruntime` package. GPU-provider validation belongs to the later portable GPU profile.

In [ ]:
import json

import yaml

with REPORT_PATH.open(encoding="utf-8") as file:
    report = json.load(file)
with (PROJECT_DIRECTORY / "configs" / "compatibility.yaml").open(encoding="utf-8") as file:
    compatibility = yaml.safe_load(file)

assert report["schema_version"] == 1
assert report["environment"] == "colab"
assert report["git"]["sha"] == ACTUAL_GIT_SHA
assert report["python"]["major_minor"] == "3.11"
assert report["packages"]["project"] == "0.1.0"
assert report["packages"]["torch"] == compatibility["pytorch"]["torch_version"]
assert (
    report["packages"]["torchvision"]
    == compatibility["pytorch"]["torchvision_version"]
)
assert report["packages"]["onnx"] == compatibility["onnx"]["version"]
assert (
    report["packages"]["onnxruntime"]
    == compatibility["onnxruntime"]["cpu_version"]
)
assert report["torch"]["cuda_available"] is True
assert report["gpu"]["device_count"] >= 1
assert report["paths"]["repository_root"]["exists"] is True
assert report["paths"]["configs"]["exists"] is True
assert report["paths"]["artifacts"]["writable"] is True
assert report["paths"]["checkpoints"]["writable"] is True
assert report["paths"]["data"]["writable"] is True
assert report["paths"]["disk"]["free_bytes"] >= 5 * 1024**3

print(json.dumps(report, indent=2, sort_keys=True))
print("\n✅ Git SHA matched")
print("✅ Required package versions matched the compatibility contract")
print("✅ CUDA available and GPU operation successful")
print("✅ Artifact, checkpoint, and data directories are writable")
print("✅ At least 5 GiB of disk space is available")
print("✅ Colab environment report verified")